# Day 057 | Bearing Capacity Prediction (ML)

## 100 Days of ML for Geotechnical Engineering
**Phase:** Classical Machine Learning

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Terzaghi Equation | Classical bearing capacity theory | Analytical baseline for comparison |
| ML vs Analytical | Data-driven vs theory-based prediction | Can ML learn nonlinear soil behavior better? |
| Feature Importance | Which soil parameters dominate capacity | Is cohesion or friction angle more critical? |
| Model Comparison | Multiple algorithms on same problem | RF vs GB vs SVR vs Linear |
| Error Analysis | Where do predictions deviate most | Extreme capacities, unusual soil conditions |
| Engineering Validation | Do ML predictions make physical sense | Capacity increases with depth, width, strength |

---

## Our Geotechnical Problem

We predict **ultimate bearing capacity (kPa)** of shallow foundations.  
The Terzaghi equation is the classical approach, but it relies on idealized assumptions.  
Can ML, trained on data, capture real-world complexity better?

---

## Part 1: Load and Explore the Dataset

In [ ]:
# ─────────────────────────────────────────────────────
# LOAD DATA AND EXPLORATION
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('bearing_capacity_ml.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSoil types: {df['soil_type'].value_counts().to_dict()}")
print(f"\nBearing capacity range: {df['bearing_capacity'].min():.0f} - {df['bearing_capacity'].max():.0f} kPa")
print(f"\nDescriptive statistics:")
print(df.describe().round(2))

In [ ]:
# ─────────────────────────────────────────────────────
# EXPLORATORY VISUALIZATIONS
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

plot_features = ['cohesion', 'friction_angle', 'footing_width',
                 'footing_depth', 'unit_weight', 'N_value']

for i, feat in enumerate(plot_features):
    row, col = i // 3, i % 3
    for soil in df['soil_type'].unique():
        mask = df['soil_type'] == soil
        axes[row, col].scatter(df.loc[mask, feat], df.loc[mask, 'bearing_capacity'],
                                alpha=0.5, s=20, label=soil)
    axes[row, col].set_xlabel(feat, fontsize=11)
    axes[row, col].set_ylabel('Bearing Capacity (kPa)', fontsize=10)
    axes[row, col].set_title(f'{feat} vs Capacity', fontsize=11, fontweight='bold')
    if i == 0:
        axes[row, col].legend(fontsize=8)

plt.suptitle('Feature Relationships with Bearing Capacity',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Part 2: Terzaghi Bearing Capacity Equation (Analytical Baseline)

In [ ]:
# ─────────────────────────────────────────────────────
# TERZAGHI EQUATION — Analytical baseline
# ─────────────────────────────────────────────────────

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def terzaghi_bearing_capacity(c, phi_deg, gamma, B, Df):
    """Terzaghi bearing capacity for strip footing."""
    phi = np.radians(phi_deg)
    # Bearing capacity factors
    Nq = np.exp(np.pi * np.tan(phi)) * np.tan(np.radians(45) + phi/2)**2
    Nc = np.where(phi_deg > 0, (Nq - 1) / (np.tan(phi) + 1e-10), 5.14)
    Ngamma = 2 * (Nq + 1) * np.tan(phi)
    # Ultimate bearing capacity
    qu = c * Nc + gamma * Df * Nq + 0.5 * gamma * B * Ngamma
    return qu

df['q_terzaghi'] = terzaghi_bearing_capacity(
    df['cohesion'].values,
    df['friction_angle'].values,
    df['unit_weight'].values,
    df['footing_width'].values,
    df['footing_depth'].values
)

r2_terz = r2_score(df['bearing_capacity'], df['q_terzaghi'])
rmse_terz = np.sqrt(mean_squared_error(df['bearing_capacity'], df['q_terzaghi']))
mae_terz = mean_absolute_error(df['bearing_capacity'], df['q_terzaghi'])

print("Terzaghi Equation Performance (on full dataset):")
print(f"  R2   = {r2_terz:.4f}")
print(f"  RMSE = {rmse_terz:.1f} kPa")
print(f"  MAE  = {mae_terz:.1f} kPa")

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(df['bearing_capacity'], df['q_terzaghi'], alpha=0.5, s=30,
           color='#3498db', edgecolors='black')
lims = [0, max(df['bearing_capacity'].max(), df['q_terzaghi'].max()) * 1.1]
ax.plot(lims, lims, 'k--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Bearing Capacity (kPa)', fontsize=12)
ax.set_ylabel('Terzaghi Prediction (kPa)', fontsize=12)
ax.set_title(f'Terzaghi Equation (R2={r2_terz:.3f})', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---

## Part 3: Prepare Data and Train ML Models

In [ ]:
# ─────────────────────────────────────────────────────
# PREPARE FEATURES AND TRAIN ML MODELS
# ─────────────────────────────────────────────────────

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               AdaBoostRegressor)
from sklearn.neighbors import KNeighborsRegressor

# Encode soil type
le = LabelEncoder()
df['soil_encoded'] = le.fit_transform(df['soil_type'])

features = ['footing_width', 'footing_length', 'footing_depth', 'soil_encoded',
            'cohesion', 'friction_angle', 'unit_weight', 'N_value', 'water_table_depth']
X = df[features]
y = df['bearing_capacity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    'Ridge': Ridge(alpha=1.0),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'SVR': SVR(kernel='rbf', C=100, epsilon=10),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=6, random_state=42)
}

results = {}
preds = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    preds[name] = y_pred
    cv = cross_val_score(model, X_train_s, y_train, cv=5, scoring='r2')
    results[name] = {
        'CV R2': cv.mean(),
        'Test R2': r2_score(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE': mean_absolute_error(y_test, y_pred)
    }

results_df = pd.DataFrame(results).T.round(4).sort_values('Test R2', ascending=False)
print("ML Model Comparison:")
print("=" * 65)
print(results_df)

---

## Part 4: ML vs Terzaghi — Head-to-Head

In [ ]:
# ─────────────────────────────────────────────────────
# ML vs TERZAGHI — Direct comparison on test set
# ─────────────────────────────────────────────────────

best_name = results_df.index[0]
y_pred_best = preds[best_name]
q_terz_test = df.loc[X_test.index, 'q_terzaghi'].values

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Terzaghi
axes[0].scatter(y_test, q_terz_test, alpha=0.6, color='#3498db',
                edgecolors='black', s=40)
axes[0].plot(lims, lims, 'k--', linewidth=2)
r2_t = r2_score(y_test, q_terz_test)
axes[0].set_title(f'Terzaghi (R2={r2_t:.3f})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual (kPa)', fontsize=12)
axes[0].set_ylabel('Predicted (kPa)', fontsize=12)

# Best ML
axes[1].scatter(y_test, y_pred_best, alpha=0.6, color='#e74c3c',
                edgecolors='black', s=40)
axes[1].plot(lims, lims, 'k--', linewidth=2)
r2_b = r2_score(y_test, y_pred_best)
axes[1].set_title(f'{best_name} (R2={r2_b:.3f})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Actual (kPa)', fontsize=12)
axes[1].set_ylabel('Predicted (kPa)', fontsize=12)

# Bar comparison
methods = ['Terzaghi', best_name]
r2_vals = [r2_t, r2_b]
rmse_vals = [np.sqrt(mean_squared_error(y_test, q_terz_test)),
             np.sqrt(mean_squared_error(y_test, y_pred_best))]

x = np.arange(len(methods))
axes[2].bar(x - 0.2, r2_vals, 0.35, label='R2', color='#2ecc71', edgecolor='black')
ax2 = axes[2].twinx()
ax2.bar(x + 0.2, rmse_vals, 0.35, label='RMSE', color='#e67e22', edgecolor='black')
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods)
axes[2].set_ylabel('R2', fontsize=12, color='#2ecc71')
ax2.set_ylabel('RMSE (kPa)', fontsize=12, color='#e67e22')
axes[2].set_title('Performance Comparison', fontsize=13, fontweight='bold')

plt.suptitle('Bearing Capacity: Terzaghi vs ML',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Part 5: Feature Importance Analysis

In [ ]:
# ─────────────────────────────────────────────────────
# FEATURE IMPORTANCE — What drives bearing capacity?
# ─────────────────────────────────────────────────────

from sklearn.inspection import permutation_importance

rf_model = models['Random Forest']
perm = permutation_importance(rf_model, X_test_s, y_test, n_repeats=20, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# MDI
sorted_mdi = np.argsort(rf_model.feature_importances_)
axes[0].barh(np.array(features)[sorted_mdi], rf_model.feature_importances_[sorted_mdi],
             color='#3498db', edgecolor='black')
axes[0].set_xlabel('MDI Importance', fontsize=12)
axes[0].set_title('Tree-Based Importance', fontsize=13, fontweight='bold')

# Permutation
sorted_perm = np.argsort(perm.importances_mean)
axes[1].barh(np.array(features)[sorted_perm], perm.importances_mean[sorted_perm],
             color='#e74c3c', edgecolor='black',
             xerr=perm.importances_std[sorted_perm])
axes[1].set_xlabel('Permutation Importance', fontsize=12)
axes[1].set_title('Permutation Importance', fontsize=13, fontweight='bold')

plt.suptitle('Feature Importance for Bearing Capacity Prediction',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Top features align with geotechnical theory:")
print("  friction_angle -> controls Nq and Ngamma (exponential effect)")
print("  cohesion       -> direct multiplier with Nc")
print("  footing_depth  -> overburden pressure increases capacity")

---

## Part 6: Error Analysis by Soil Type

In [ ]:
# ─────────────────────────────────────────────────────
# ERROR ANALYSIS — Where do models struggle?
# ─────────────────────────────────────────────────────

test_soils = df.loc[X_test.index, 'soil_type'].values
residuals_terz = y_test.values - q_terz_test
residuals_ml = y_test.values - y_pred_best

error_df = pd.DataFrame({
    'soil_type': test_soils,
    'actual': y_test.values,
    'Terzaghi_error': np.abs(residuals_terz),
    'ML_error': np.abs(residuals_ml)
})

error_summary = error_df.groupby('soil_type').agg({
    'actual': 'mean',
    'Terzaghi_error': ['mean', 'std'],
    'ML_error': ['mean', 'std']
}).round(1)

print("Error Analysis by Soil Type:")
print("=" * 60)
print(error_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

soil_groups = error_df.groupby('soil_type')
soil_names = sorted(error_df['soil_type'].unique())

# MAPE by soil type
terz_mape = [error_df.loc[error_df['soil_type']==s, 'Terzaghi_error'].mean() /
             error_df.loc[error_df['soil_type']==s, 'actual'].mean() * 100 for s in soil_names]
ml_mape = [error_df.loc[error_df['soil_type']==s, 'ML_error'].mean() /
           error_df.loc[error_df['soil_type']==s, 'actual'].mean() * 100 for s in soil_names]

x = np.arange(len(soil_names))
axes[0].bar(x - 0.2, terz_mape, 0.35, label='Terzaghi', color='#3498db', edgecolor='black')
axes[0].bar(x + 0.2, ml_mape, 0.35, label='ML', color='#e74c3c', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(soil_names)
axes[0].set_ylabel('MAPE (%)', fontsize=12)
axes[0].set_title('Mean Absolute % Error by Soil Type', fontsize=13, fontweight='bold')
axes[0].legend()

# Residual distributions
axes[1].hist(residuals_terz, bins=20, alpha=0.5, color='#3498db', edgecolor='black',
             label=f'Terzaghi (std={residuals_terz.std():.0f})')
axes[1].hist(residuals_ml, bins=20, alpha=0.5, color='#e74c3c', edgecolor='black',
             label=f'ML (std={residuals_ml.std():.0f})')
axes[1].set_xlabel('Residual (kPa)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Residual Distribution', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

---

## Part 7: Engineering Validation — Do Predictions Make Physical Sense?

In [ ]:
# ─────────────────────────────────────────────────────
# ENGINEERING VALIDATION — Monotonicity checks
# ─────────────────────────────────────────────────────

# Create synthetic scenarios to check if ML respects physics
best_model = models[best_name]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Increasing friction angle -> increasing capacity
phi_range = np.linspace(5, 40, 50)
base_vals = X_train.median().values
X_phi = np.tile(base_vals, (50, 1))
phi_idx = features.index('friction_angle')
X_phi[:, phi_idx] = phi_range
X_phi_s = scaler.transform(X_phi)
pred_phi = best_model.predict(X_phi_s)

axes[0].plot(phi_range, pred_phi, 'b-o', linewidth=2, markersize=3)
axes[0].set_xlabel('Friction Angle (degrees)', fontsize=12)
axes[0].set_ylabel('Predicted Capacity (kPa)', fontsize=12)
axes[0].set_title('Effect of Friction Angle', fontsize=12, fontweight='bold')

# 2. Increasing footing width -> increasing capacity
B_range = np.linspace(1, 5, 50)
X_B = np.tile(base_vals, (50, 1))
B_idx = features.index('footing_width')
X_B[:, B_idx] = B_range
X_B_s = scaler.transform(X_B)
pred_B = best_model.predict(X_B_s)

axes[1].plot(B_range, pred_B, 'r-o', linewidth=2, markersize=3)
axes[1].set_xlabel('Footing Width (m)', fontsize=12)
axes[1].set_ylabel('Predicted Capacity (kPa)', fontsize=12)
axes[1].set_title('Effect of Footing Width', fontsize=12, fontweight='bold')

# 3. Increasing depth -> increasing capacity
D_range = np.linspace(0.5, 3, 50)
X_D = np.tile(base_vals, (50, 1))
D_idx = features.index('footing_depth')
X_D[:, D_idx] = D_range
X_D_s = scaler.transform(X_D)
pred_D = best_model.predict(X_D_s)

axes[2].plot(D_range, pred_D, 'g-o', linewidth=2, markersize=3)
axes[2].set_xlabel('Footing Depth (m)', fontsize=12)
axes[2].set_ylabel('Predicted Capacity (kPa)', fontsize=12)
axes[2].set_title('Effect of Footing Depth', fontsize=12, fontweight='bold')

plt.suptitle('Engineering Validation: Does ML Respect Physics?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Check: Capacity should generally INCREASE with:")
print("  - Higher friction angle (stronger soil)")
print("  - Wider footing (more bearing area)")
print("  - Deeper footing (more overburden)")
print("If ML violates these trends, the model may not be physically valid.")

---

## Practice Exercises

### Exercise 1: Shape Factor Correction
Modify the Terzaghi equation to include shape factors for rectangular footings. Does the improved equation close the gap with ML?

### Exercise 2: Ensemble ML
Create a stacking ensemble (RF + GB + SVR). Does it beat the best individual model?

### Exercise 3: Safety Factor
Apply a safety factor of 3 to both Terzaghi and ML predictions. For how many test samples does each method give allowable capacity within 20% of the true value?

In [ ]:
# EXERCISE 3 STARTER:
# FoS = 3.0
# q_allow_terz = q_terz_test / FoS
# q_allow_ml = y_pred_best / FoS
# q_allow_actual = y_test.values / FoS
# 
# within_20_terz = np.sum(np.abs(q_allow_terz - q_allow_actual) / q_allow_actual < 0.20)
# within_20_ml = np.sum(np.abs(q_allow_ml - q_allow_actual) / q_allow_actual < 0.20)
# print(f"Terzaghi within 20%: {within_20_terz}/{len(y_test)}")
# print(f"ML within 20%:       {within_20_ml}/{len(y_test)}")

---

## Key Takeaways

1. **Terzaghi equation** provides a solid analytical baseline but assumes idealized conditions (homogeneous soil, simple geometry).

2. **ML models** can capture nonlinear interactions and real-world complexity that analytical equations miss.

3. **Feature importance** confirms that friction angle and cohesion dominate — consistent with geotechnical theory.

4. **Engineering validation** is critical: ML predictions must respect physical trends (more strength = more capacity).

5. **Error analysis by soil type** helps identify where models struggle and may need more training data.

6. ML should **complement, not replace** classical methods. Use both and investigate disagreements.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| Terzaghi equation | Physics-based baseline model |
| Data-driven prediction | Supervised regression |
| Feature importance | Understanding what the model learned |
| Monotonicity check | Domain-knowledge validation |
| Error by category | Stratified evaluation |
| ML + theory comparison | Hybrid modeling approach |

---

*Day 057 of 100 — Theory meets data: the best of both worlds!*